# Price Impact HTE Analysis Pipeline

This notebook consolidates the full **Heterogeneous Treatment Effect (HTE)** analysis for estimating price impact of trade size on market microstructure data.

**Pipeline Steps:**
1. Configuration & Imports
2. Causal DAG & Assumptions
3. Synthetic Data Generation (DGP)
4. Feature Engineering
5. HTE Estimator Fitting
6. Evaluation & Metrics
7. Visualization

---
## 1. Configuration & Imports

In [ ]:
# Core libraries for data, math, and ML
import time
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from scipy import stats as sp_stats

# Visualization
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ML models
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# Causal ML
from econml.dml import CausalForestDML
from econml.metalearners import SLearner, TLearner, XLearner, DRLearner

# Render plots inline
%matplotlib inline

print("All imports loaded successfully.")

In [ ]:
# Inline configuration (equivalent to config/params.yaml)
CONFIG = {
    "random_seed": 42,
    "dgp": {
        "n_observations": 200000,
        "train_fraction": 0.7,
        # Covariate distributions
        "spread_loc": -2.0,
        "spread_scale": 0.6,
        "depth_scale": 50.0,
        "volatility_scale": 0.02,
        # Treatment (confounded by market state)
        "treatment_noise_scale": 5.0,
        "treatment_depth_coef": 0.3,
        "treatment_vol_coef": -10.0,
        # Outcome — true CATE: τ(X) = β₀ + β₁/depth + β₂·vol
        "baseline_drift_scale": 0.001,
        "cate_intercept": 0.0005,
        "cate_depth_coef": 0.5,
        "cate_vol_coef": 0.05,
        "outcome_noise_scale": 0.002,
        # AR(1) autocorrelation in noise (0 = IID)
        "ar1_coefficient": 0.3,
    },
    "features": {
        "volatility_window": 20,
        "winsorize_percentile": 0.01,
    },
    "estimators": {
        "causal_forest": {
            "n_estimators": 1000,
            "min_samples_leaf": 20,
            "max_depth": None,
            "discrete_treatment": False,
        },
        "s_learner": {"base_model": "GradientBoostingRegressor", "n_estimators": 200},
        "t_learner": {"base_model": "GradientBoostingRegressor", "n_estimators": 200},
        "x_learner": {"base_model": "GradientBoostingRegressor", "n_estimators": 200},
        "dr_learner": {"base_model": "GradientBoostingRegressor", "n_estimators": 200},
        "binary_treatment_quantile": 0.75,
    },
    "evaluation": {
        "n_bootstrap": 200,
        "ci_level": 0.90,
        "n_cate_bins": 5,
    },
    "output": {
        "results_dir": "results",
        "figure_format": "png",
        "figure_dpi": 300,
    },
}

cfg = CONFIG
print("Configuration loaded.")

---
## 2. Causal DAG & Identifying Assumptions

In [ ]:
# Causal directed acyclic graph in DOT format
CAUSAL_GRAPH_DOT = """
digraph {
    spread -> trade_size;
    depth -> trade_size;
    volatility -> trade_size;

    spread -> price_change;
    depth -> price_change;
    volatility -> price_change;

    trade_size -> price_change;
}
"""

# Print the identifying assumptions for the causal model
ASSUMPTIONS = """
1. UNCONFOUNDEDNESS: Conditional on (spread, depth, volatility), trade_size
   is as-good-as-random. No unobserved confounders affect both T and Y.

2. OVERLAP (POSITIVITY): For every market state X, there is positive
   probability of observing any trade size T.

3. SUTVA: One trade's outcome does not depend on other trades' treatment
   assignments.

4. CONSISTENCY: The observed outcome under treatment T=t equals the
   potential outcome Y(t).
"""


def get_dowhy_graph() -> str:
    """Return the causal graph in DOT format."""
    return CAUSAL_GRAPH_DOT


def build_dowhy_model(df, treatment="trade_size", outcome="price_change",
                       common_causes=None):
    """Build a DoWhy CausalModel for refutation tests."""
    try:
        from dowhy import CausalModel
    except ImportError:
        print("dowhy not installed — skipping causal model construction.")
        return None

    if common_causes is None:
        common_causes = ["spread", "depth", "volatility"]

    model = CausalModel(
        data=df,
        treatment=treatment,
        outcome=outcome,
        common_causes=common_causes,
        graph=get_dowhy_graph(),
    )
    return model


print("=== Causal Graph ===")
print(get_dowhy_graph())
print(ASSUMPTIONS)

---
## 3. Synthetic Data Generation (DGP)

In [ ]:
def compute_true_cate(depth, volatility, cfg):
    """Compute ground-truth CATE: τ(X) = β₀ + β₁/depth + β₂·volatility."""
    dgp = cfg["dgp"]
    cate = (
        dgp["cate_intercept"]
        + dgp["cate_depth_coef"] / (depth + 1e-6)
        + dgp["cate_vol_coef"] * volatility
    )
    return cate


def generate_data(cfg, seed=None):
    """Generate synthetic trade data with heterogeneous price impact."""
    if seed is None:
        seed = cfg["random_seed"]
    rng = np.random.default_rng(seed)
    dgp = cfg["dgp"]
    n = dgp["n_observations"]

    # Market-state covariates
    spread = rng.lognormal(mean=dgp["spread_loc"], sigma=dgp["spread_scale"], size=n)
    depth = rng.exponential(scale=dgp["depth_scale"], size=n)
    volatility = np.abs(rng.normal(0, dgp["volatility_scale"], size=n))

    # Treatment (trade size) — confounded by market state
    treatment_mean = (
        dgp["treatment_depth_coef"] * depth
        + dgp["treatment_vol_coef"] * volatility
    )
    treatment_noise = rng.normal(0, dgp["treatment_noise_scale"], size=n)
    trade_size = np.maximum(treatment_mean + treatment_noise, 1.0)

    # Ground-truth CATE
    true_cate = compute_true_cate(depth, volatility, cfg)

    # Outcome: Y = g(X) + τ(X)·T + ε
    baseline_drift = -dgp["baseline_drift_scale"] * spread

    # Noise with optional AR(1) autocorrelation
    iid_noise = rng.normal(0, dgp["outcome_noise_scale"], size=n)
    ar1 = dgp.get("ar1_coefficient", 0.0)
    if ar1 != 0.0:
        noise = np.zeros(n)
        noise[0] = iid_noise[0]
        for t_idx in range(1, n):
            noise[t_idx] = ar1 * noise[t_idx - 1] + iid_noise[t_idx]
    else:
        noise = iid_noise

    price_change = baseline_drift + true_cate * trade_size + noise

    df = pd.DataFrame({
        "spread": spread,
        "depth": depth,
        "volatility": volatility,
        "trade_size": trade_size,
        "price_change": price_change,
        "true_cate": true_cate,
    })

    return df


def split_data(df, cfg):
    """Temporal train/test split to respect time ordering."""
    frac = cfg["dgp"]["train_fraction"]
    split_idx = int(len(df) * frac)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

In [ ]:
# Generate data and split into train/test
t0 = time.time()

print("=" * 60)
print("  Price Impact HTE Analysis Pipeline")
print("=" * 60)

print("\n[1/6] Generating synthetic data...")
df = generate_data(cfg)
train_raw, test_raw = split_data(df, cfg)
print(f"  Total: {len(df):,}  |  Train: {len(train_raw):,}  |  Test: {len(test_raw):,}")

# Quick data summary
print(f"\nCovariate summary:")
display(df[['spread', 'depth', 'volatility']].describe().round(4))
print(f"\nTreatment (trade_size):")
display(df['trade_size'].describe().round(4))
print(f"\nTrue CATE:")
display(df['true_cate'].describe().round(6))

---
## 4. Feature Engineering

In [ ]:
def winsorize(series, pct=0.01):
    """Clip extreme values at the given percentile on both tails."""
    lower = series.quantile(pct)
    upper = series.quantile(1 - pct)
    return series.clip(lower, upper)


def add_rolling_volatility(df, window=20):
    """Backward-looking rolling standard deviation of price changes."""
    df = df.copy()
    df["rolling_vol"] = (
        df["price_change"]
        .rolling(window=window, min_periods=1)
        .std()
        .bfill()
    )
    return df


def add_liquidity_features(df):
    """Derive log_spread, inv_depth, and spread_depth_ratio."""
    df = df.copy()
    df["log_spread"] = np.log1p(df["spread"])
    df["inv_depth"] = 1.0 / (df["depth"] + 1e-6)
    df["spread_depth_ratio"] = df["spread"] / (df["depth"] + 1e-6)
    return df


def standardize(df, columns, fit_stats=None):
    """Z-score standardize columns. Reuses fit_stats from training if provided."""
    df = df.copy()
    if fit_stats is None:
        fit_stats = {}
        for col in columns:
            fit_stats[col] = {"mean": df[col].mean(), "std": df[col].std()}

    for col in columns:
        mu = fit_stats[col]["mean"]
        sigma = fit_stats[col]["std"]
        df[col] = (df[col] - mu) / (sigma + 1e-10)

    return df, fit_stats


def engineer_features(df, cfg, fit_stats=None):
    """Full pipeline: winsorize → derive features → standardize."""
    feat_cfg = cfg["features"]

    pct = feat_cfg["winsorize_percentile"]
    for col in ["spread", "depth", "volatility", "trade_size"]:
        df[col] = winsorize(df[col], pct)

    df = add_liquidity_features(df)
    df = add_rolling_volatility(df, window=feat_cfg["volatility_window"])

    covariate_cols = [
        "spread", "depth", "volatility",
        "log_spread", "inv_depth", "spread_depth_ratio", "rolling_vol"
    ]
    df, fit_stats = standardize(df, covariate_cols, fit_stats)

    return df, fit_stats

In [ ]:
# Apply feature engineering to train and test sets
print("[2/6] Engineering features...")
train, fit_stats = engineer_features(train_raw, cfg)
test, _ = engineer_features(test_raw, cfg, fit_stats=fit_stats)
print(f"  Features: {list(train.columns)}")

---
## 5. HTE Estimators

In [ ]:
def _get_base_model(model_name, n_estimators=200):
    """Instantiate a scikit-learn base model by name."""
    models = {
        "GradientBoostingRegressor": GradientBoostingRegressor(
            n_estimators=n_estimators, max_depth=4, random_state=42
        ),
        "RandomForestRegressor": RandomForestRegressor(
            n_estimators=n_estimators, max_depth=10, random_state=42
        ),
    }
    return models.get(model_name, GradientBoostingRegressor(
        n_estimators=n_estimators, random_state=42
    ))


def binarize_treatment(trade_size, quantile=0.75):
    """Convert continuous trade size to binary (1 if above quantile threshold)."""
    threshold = np.quantile(trade_size, quantile)
    return (trade_size >= threshold).astype(int)


class EstimatorSuite:
    """Fits and evaluates all five HTE estimators."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.estimators = {}
        self.fitted = {}

    def fit_all(self, Y, T, X):
        """Fit all estimators on the training data."""
        est_cfg = self.cfg["estimators"]
        T_binary = binarize_treatment(T, est_cfg["binary_treatment_quantile"])

        self._fit_causal_forest(Y, T, X, est_cfg["causal_forest"])
        self._fit_s_learner(Y, T_binary, X, est_cfg["s_learner"])
        self._fit_t_learner(Y, T_binary, X, est_cfg["t_learner"])
        self._fit_x_learner(Y, T_binary, X, est_cfg["x_learner"])
        self._fit_dr_learner(Y, T_binary, X, est_cfg["dr_learner"])
        return self

    def _fit_causal_forest(self, Y, T, X, params):
        """Fit CausalForestDML (continuous treatment)."""
        est = CausalForestDML(
            model_y=RandomForestRegressor(n_estimators=100, random_state=42),
            model_t=RandomForestRegressor(n_estimators=100, random_state=42),
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_depth=params.get("max_depth"),
            discrete_treatment=params["discrete_treatment"],
            random_state=42,
        )
        est.fit(Y, T, X=X)
        self.fitted["CausalForest"] = est
        print("  ✓ CausalForestDML fitted")

    def _fit_s_learner(self, Y, T, X, params):
        """Fit S-Learner (single model, treatment as feature)."""
        base = _get_base_model(params["base_model"], params["n_estimators"])
        est = SLearner(overall_model=base)
        est.fit(Y, T, X=X)
        self.fitted["S-Learner"] = est
        print("  ✓ S-Learner fitted")

    def _fit_t_learner(self, Y, T, X, params):
        """Fit T-Learner (separate models per treatment arm)."""
        base = _get_base_model(params["base_model"], params["n_estimators"])
        est = TLearner(models=base)
        est.fit(Y, T, X=X)
        self.fitted["T-Learner"] = est
        print("  ✓ T-Learner fitted")

    def _fit_x_learner(self, Y, T, X, params):
        """Fit X-Learner (cross-fitted imputed effects)."""
        base = _get_base_model(params["base_model"], params["n_estimators"])
        est = XLearner(models=base)
        est.fit(Y, T, X=X)
        self.fitted["X-Learner"] = est
        print("  ✓ X-Learner fitted")

    def _fit_dr_learner(self, Y, T, X, params):
        """Fit DR-Learner (doubly robust CATE estimation)."""
        base = _get_base_model(params["base_model"], params["n_estimators"])
        est = DRLearner(models=base)
        est.fit(Y, T, X=X)
        self.fitted["DR-Learner"] = est
        print("  ✓ DR-Learner fitted")

    def estimate_cate(self, X):
        """Predict CATE for all fitted estimators."""
        results = {}
        for name, est in self.fitted.items():
            cate = est.effect(X)
            results[name] = cate.flatten()
        return results

    def confidence_intervals(self, X, alpha=0.1):
        """Get confidence intervals where supported."""
        ci_results = {}
        for name, est in self.fitted.items():
            if hasattr(est, "effect_interval"):
                try:
                    lower, upper = est.effect_interval(X, alpha=alpha)
                    ci_results[name] = (lower.flatten(), upper.flatten())
                except Exception:
                    pass
        return ci_results

In [ ]:
# Prepare arrays and fit all estimators
print("[3/6] Fitting HTE estimators...")
covariate_cols = ["spread", "depth", "volatility"]
X_train = train[covariate_cols].values
X_test = test[covariate_cols].values
Y_train = train["price_change"].values
T_train = train["trade_size"].values
true_cate_test = test["true_cate"].values

suite = EstimatorSuite(cfg)
suite.fit_all(Y_train, T_train, X_train)

In [ ]:
# Estimate CATE and confidence intervals on test set
print("[4/6] Estimating CATE on test set...")
cate_dict = suite.estimate_cate(X_test)
ci_dict = suite.confidence_intervals(X_test, alpha=1 - cfg["evaluation"]["ci_level"])
print("  Done.")

---
## 6. Evaluation & Metrics

In [ ]:
def cate_rmse(estimated, true):
    """Root mean squared error between estimated and true CATE."""
    return np.sqrt(np.mean((estimated - true) ** 2))


def cate_mae(estimated, true):
    """Mean absolute error between estimated and true CATE."""
    return np.mean(np.abs(estimated - true))


def cate_bias(estimated, true):
    """Mean bias (positive = overestimation)."""
    return np.mean(estimated - true)


def rank_correlation(estimated, true):
    """Spearman and Kendall rank correlations between estimated and true CATE."""
    spearman_r, spearman_p = sp_stats.spearmanr(estimated, true)
    kendall_tau, kendall_p = sp_stats.kendalltau(estimated, true)
    return {
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
        "kendall_tau": kendall_tau,
        "kendall_p": kendall_p,
    }


def ci_coverage(true, lower, upper):
    """Fraction of true CATEs falling inside the confidence interval."""
    covered = (true >= lower) & (true <= upper)
    return np.mean(covered)


def gates_analysis(estimated, true, n_bins=5):
    """Bin by estimated CATE quintile, compare average estimated vs true."""
    bin_labels = pd.qcut(estimated, q=n_bins, labels=False, duplicates="drop")
    df_gates = pd.DataFrame({
        "estimated": estimated,
        "true": true,
        "bin": bin_labels,
    })
    summary = df_gates.groupby("bin").agg(
        mean_estimated=("estimated", "mean"),
        mean_true=("true", "mean"),
        count=("estimated", "count"),
    ).reset_index()
    summary["abs_error"] = np.abs(summary["mean_estimated"] - summary["mean_true"])
    return summary


def evaluate_all(cate_dict, true_cate, ci_dict=None, n_bins=5):
    """Run all metrics for every estimator, return summary DataFrame."""
    rows = []
    for name, est_cate in cate_dict.items():
        row = {"Estimator": name}
        row["RMSE"] = cate_rmse(est_cate, true_cate)
        row["MAE"] = cate_mae(est_cate, true_cate)
        row["Bias"] = cate_bias(est_cate, true_cate)

        rank_corr = rank_correlation(est_cate, true_cate)
        row["Spearman_r"] = rank_corr["spearman_r"]
        row["Kendall_tau"] = rank_corr["kendall_tau"]

        if ci_dict and name in ci_dict:
            lower, upper = ci_dict[name]
            row["CI_Coverage"] = ci_coverage(true_cate, lower, upper)
        else:
            row["CI_Coverage"] = np.nan

        rows.append(row)

    results = pd.DataFrame(rows)
    results = results.set_index("Estimator")
    return results

In [ ]:
# Run evaluation
print("[5/6] Evaluating estimators...")
results_df = evaluate_all(cate_dict, true_cate_test, ci_dict,
                           n_bins=cfg["evaluation"]["n_cate_bins"])

print("\n" + "=" * 60)
print("  ESTIMATOR PERFORMANCE SUMMARY")
print("=" * 60)
display(results_df.style.format("{:.6f}"))

# GATES analysis per estimator
gates_dict = {}
for name, est_cate in cate_dict.items():
    gates_dict[name] = gates_analysis(
        est_cate, true_cate_test, n_bins=cfg["evaluation"]["n_cate_bins"]
    )

---
## 7. Visualization

In [ ]:
def set_style():
    """Set publication-ready matplotlib style."""
    plt.style.use("seaborn-v0_8-whitegrid")
    mpl.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


def plot_cate_heatmap(df, cate_col, title, save_path=None):
    """2D heatmap of CATE across depth and volatility bins."""
    set_style()
    fig, ax = plt.subplots(figsize=(8, 6))

    df_plot = df.copy()
    df_plot["depth_bin"] = pd.qcut(df_plot["depth"], q=20, duplicates="drop")
    df_plot["vol_bin"] = pd.qcut(df_plot["volatility"], q=20, duplicates="drop")

    pivot = df_plot.groupby(["vol_bin", "depth_bin"])[cate_col].mean().unstack()
    sns.heatmap(pivot, cmap="RdYlBu_r", center=0, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Order Book Depth (binned)")
    ax.set_ylabel("Volatility (binned)")

    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig


def plot_cate_scatter(estimated, true, estimator_name, save_path=None):
    """Scatter plot of estimated vs true CATE with 45° reference line."""
    set_style()
    fig, ax = plt.subplots(figsize=(7, 7))

    n = len(true)
    idx = np.random.choice(n, size=min(5000, n), replace=False)

    ax.scatter(true[idx], estimated[idx], alpha=0.3, s=8, color="#2196F3")
    lims = [
        min(true[idx].min(), estimated[idx].min()),
        max(true[idx].max(), estimated[idx].max()),
    ]
    ax.plot(lims, lims, "k--", alpha=0.7, label="Perfect estimation")
    ax.set_xlabel("True CATE")
    ax.set_ylabel("Estimated CATE")
    ax.set_title(f"{estimator_name}: Estimated vs True CATE")
    ax.legend()

    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig


def plot_estimator_comparison(results_df, metric="RMSE", save_path=None):
    """Horizontal bar chart comparing estimators on a given metric."""
    set_style()
    fig, ax = plt.subplots(figsize=(9, 5))

    colors = sns.color_palette("viridis", n_colors=len(results_df))
    results_df[metric].plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_xlabel(metric)
    ax.set_title(f"Estimator Comparison — {metric}")
    ax.invert_yaxis()

    for i, v in enumerate(results_df[metric]):
        ax.text(v + 0.0001, i, f"{v:.5f}", va="center", fontsize=9)

    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig


def plot_gates(gates_df, estimator_name, save_path=None):
    """Bar chart of mean estimated vs true CATE by quintile bin."""
    set_style()
    fig, ax = plt.subplots(figsize=(8, 5))

    x = gates_df["bin"]
    width = 0.35
    ax.bar(x - width / 2, gates_df["mean_estimated"], width,
           label="Estimated", color="#4CAF50", alpha=0.8)
    ax.bar(x + width / 2, gates_df["mean_true"], width,
           label="True", color="#FF5722", alpha=0.8)

    ax.set_xlabel("CATE Quintile Bin")
    ax.set_ylabel("Mean CATE")
    ax.set_title(f"GATES Analysis — {estimator_name}")
    ax.legend()
    ax.set_xticks(x)

    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig


def plot_partial_dependence(X, cate, feature_names, save_path=None):
    """Binned average CATE as a function of each covariate."""
    set_style()
    n_features = min(len(feature_names), X.shape[1])
    fig, axes = plt.subplots(1, n_features, figsize=(5 * n_features, 4))
    if n_features == 1:
        axes = [axes]

    for i, (ax, fname) in enumerate(zip(axes, feature_names[:n_features])):
        bins = pd.qcut(X[:, i], q=20, duplicates="drop")
        df_tmp = pd.DataFrame({"feature": X[:, i], "cate": cate, "bin": bins})
        agg = df_tmp.groupby("bin")["cate"].mean()

        ax.plot(range(len(agg)), agg.values, "o-", color="#673AB7", markersize=4)
        ax.set_xlabel(fname)
        ax.set_ylabel("Mean Estimated CATE")
        ax.set_title(f"Partial Dependence: {fname}")
        ax.tick_params(axis="x", rotation=45)
        ax.set_xticks([])

    fig.suptitle("Partial Dependence of CATE on Market-State Features", y=1.02)
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig

In [ ]:
# Generate all plots
print("[6/6] Generating plots...\n")

out_dir = Path(cfg["output"]["results_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
fmt = cfg["output"]["figure_format"]

In [ ]:
# True CATE heatmap
plot_cate_heatmap(
    test_raw, "true_cate", "True CATE (Ground Truth)",
    save_path=str(out_dir / f"cate_heatmap_true.{fmt}")
);

In [ ]:
# Estimated vs True CATE scatter plots for each estimator
for name, est_cate in cate_dict.items():
    safe_name = name.replace(" ", "_").replace("-", "_").lower()
    plot_cate_scatter(
        est_cate, true_cate_test, name,
        save_path=str(out_dir / f"scatter_{safe_name}.{fmt}")
    )

In [ ]:
# Estimator comparison bar charts
plot_estimator_comparison(
    results_df, "RMSE",
    save_path=str(out_dir / f"estimator_comparison_rmse.{fmt}")
)
plot_estimator_comparison(
    results_df, "MAE",
    save_path=str(out_dir / f"estimator_comparison_mae.{fmt}")
);

In [ ]:
# Partial dependence plots for the best estimator
feature_names = ["spread", "depth", "volatility"]
X_test_raw = test_raw[feature_names].values
best = results_df["RMSE"].idxmin()

plot_partial_dependence(
    X_test_raw, cate_dict[best], feature_names,
    save_path=str(out_dir / f"partial_dependence_{best.lower().replace('-','_')}.{fmt}")
);

In [ ]:
# GATES analysis plots per estimator
for name, gates_df in gates_dict.items():
    safe_name = name.replace(" ", "_").replace("-", "_").lower()
    plot_gates(
        gates_df, name,
        save_path=str(out_dir / f"gates_{safe_name}.{fmt}")
    )

In [ ]:
# Save results table to CSV
results_df.to_csv(out_dir / "estimator_results.csv")

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  Pipeline complete in {elapsed:.1f}s")
print(f"  Results saved to {out_dir}/")
print(f"{'=' * 60}")